# Customer Churn Prediction
## Machine Learning Pipeline with SVM

Complete ML pipeline to predict customer churn using SVM with SMOTE for class imbalance.

## 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE

np.random.seed(42)
print("✓ All libraries imported successfully")

## 2. Load Data

In [ ]:
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"✓ Dataset loaded: {df.shape}")
df.head()

## 3. Data Preprocessing

In [ ]:
# Remove customerID and handle missing values
df.drop("customerID", axis=1, inplace=True)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)

print(f"✓ Cleaning complete: {df.shape}")
print(f"Churn distribution:\n{df['Churn'].value_counts()}")

In [ ]:
# Encode categorical features
le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == "object":
        df[col] = le.fit_transform(df[col])

print(f"✓ Encoding complete: All {df.shape[1]} features are numeric")

## 4. Train-Test Split & Scaling

In [ ]:
# Split data
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features for SVM
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Train-test split & scaling complete")
print(f"  - Training: {X_train_scaled.shape}")
print(f"  - Testing: {X_test_scaled.shape}")

## 5. SMOTE Class Balancing

In [ ]:
# Apply SMOTE to balance training data
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"✓ SMOTE resampling complete")
print(f"  - Original shape: {X_train_scaled.shape}")
print(f"  - Balanced shape: {X_train_balanced.shape}")
print(f"  - Original classes: {np.bincount(y_train)}")
print(f"  - Balanced classes: {np.bincount(y_train_balanced)}")

## 6. Train SVM Model

In [ ]:
# Initialize SVM with probability calibration
svm_model = SVC(
    probability=True,          # Enable probability predictions
    class_weight='balanced',   # Handle remaining class imbalance
    random_state=42,           # Reproducibility
    kernel='rbf'               # Radial basis function
)

# Train on balanced data
svm_model.fit(X_train_balanced, y_train_balanced)

print(f"✓ SVM model trained successfully")
print(f"  - Kernel: RBF")
print(f"  - Support vectors: {len(svm_model.support_vectors_)}")

## 7. Model Evaluation with Threshold Tuning

In [ ]:
# Make predictions with probabilities
y_pred_default = svm_model.predict(X_test_scaled)
y_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

print(f"✓ Predictions generated")
print(f"  - Probability range: [{y_proba.min():.4f}, {y_proba.max():.4f}]")

In [ ]:
# Evaluation: Default threshold (0.5)
print("="*60)
print("SVM - DEFAULT THRESHOLD (0.5)")
print("="*60)
print(classification_report(y_test, y_pred_default, target_names=['No Churn', 'Churn']))

In [ ]:
# Evaluation: Custom threshold 0.65 (balanced)
y_pred_065 = (y_proba >= 0.65).astype(int)

print("="*60)
print("SVM - THRESHOLD 0.65 (Balanced)")
print("="*60)
print(classification_report(y_test, y_pred_065, target_names=['No Churn', 'Churn']))

In [ ]:
# Evaluation: Custom threshold 0.7 (higher precision)
y_pred_070 = (y_proba >= 0.7).astype(int)

print("="*60)
print("SVM - CUSTOM THRESHOLD 0.7 (Higher Precision)")
print("="*60)
print(classification_report(y_test, y_pred_070, target_names=['No Churn', 'Churn']))

## 8. Confusion Matrix & Visualization

In [ ]:
# Create images directory if needed
os.makedirs("images", exist_ok=True)

# Generate confusion matrix for threshold 0.7
cm = confusion_matrix(y_test, y_pred_070)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=True,
    ax=ax,
    xticklabels=['No Churn', 'Churn'],
    yticklabels=['No Churn', 'Churn'],
    annot_kws={'size': 14, 'weight': 'bold'}
)
ax.set_xlabel("Predicted Label", fontsize=12, fontweight='bold')
ax.set_ylabel("True Label", fontsize=12, fontweight='bold')
ax.set_title("Confusion Matrix - SVM with Threshold 0.7", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()

# Save figure
output_path = "images/final_model.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"✓ Confusion matrix saved to: {output_path}")

# Display
plt.show()

# Print statistics
print(f"\nConfusion Matrix:")
tn, fp, fn, tp = cm.ravel()
print(f"  TN: {tn} | FP: {fp}")
print(f"  FN: {fn} | TP: {tp}")

## 9. Final Summary

In [ ]:
# Calculate final metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_070),
    'Precision': precision_score(y_test, y_pred_070),
    'Recall': recall_score(y_test, y_pred_070),
    'F1-Score': f1_score(y_test, y_pred_070)
}

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE (Threshold: 0.7)")
print("="*60)
for metric, value in metrics.items():
    print(f"{metric:.<40} {value:.4f}")
print("="*60)
print(f"\n✓ Model successfully trained and evaluated!")
print(f"✓ Results saved to images/final_model.png")

# Customer Churn Prediction
## Machine Learning Pipeline with SVM Model

This notebook implements a complete ML pipeline to predict customer churn using Support Vector Machine (SVM) with SMOTE for handling class imbalance.

## 1. Import Required Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE

# Set random seeds for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully")

## 2. Load the Data

In [ ]:
# Load customer churn dataset
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(f"✓ Dataset loaded successfully")
print(f"  - Shape: {df.shape}")
print(f"  - Columns: {df.shape[1]}")
print(f"  - Rows: {df.shape[0]}")
print("\nFirst 5 rows:")
df.head()

## 3. Data Preprocessing

In [ ]:
# Step 1: Remove customerID column (not useful for prediction)
df.drop("customerID", axis=1, inplace=True)

# Step 2: Convert TotalCharges to numeric (handles non-numeric values)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Step 3: Remove rows with missing values
df.dropna(inplace=True)

print(f"✓ Data cleaning completed")
print(f"  - Dataset shape after cleaning: {df.shape}")
print(f"\nChurn distribution:")
print(df["Churn"].value_counts())
print(f"\nClass balance:")
print(df["Churn"].value_counts(normalize=True))

In [ ]:
# Encode all categorical variables to numeric format
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = le.fit_transform(df[col])

print(f"✓ Categorical encoding completed")
print(f"  - All {df.shape[1]} features are now numeric")
print("\nProcessed dataset (first 5 rows):")
df.head()

## 4. Train-Test Split

In [ ]:
# Separate features (X) and target (y)
X = df.drop("Churn", axis=1)
y = df["Churn"]

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Train-test split completed")
print(f"  - Training set: {X_train.shape}")
print(f"  - Testing set: {X_test.shape}")
print(f"  - Feature count: {X_train.shape[1]}")

In [ ]:
# Scale features using StandardScaler (required for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✓ Feature scaling completed")
print(f"  - Training set mean: {X_train_scaled.mean():.4f}")
print(f"  - Training set std: {X_train_scaled.std():.4f}")

## 5. Handle Class Imbalance with SMOTE

In [ ]:
# Apply SMOTE (Synthetic Minority Over-sampling Technique) to balance classes
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"✓ SMOTE resampling completed")
print(f"\nOriginal training set:")
print(f"  - Shape: {X_train_scaled.shape}")
print(f"  - Class distribution: {np.bincount(y_train)}")
print(f"\nBalanced training set:")
print(f"  - Shape: {X_train_balanced.shape}")
print(f"  - Class distribution: {np.bincount(y_train_balanced)}")

## 6. Train SVM Model

In [ ]:
# Initialize and train Support Vector Machine classifier
svm_model = SVC(
    probability=True,           # Enable probability estimates
    class_weight='balanced',    # Penalize minority class less
    random_state=42,            # Reproducibility
    kernel='rbf',               # Radial basis function kernel
    C=1.0                       # Regularization parameter
)

# Fit model on balanced training data
svm_model.fit(X_train_balanced, y_train_balanced)

print(f"✓ SVM model trained successfully")
print(f"  - Kernel: RBF")
print(f"  - Support vectors: {len(svm_model.support_vectors_)}")
print(f"  - Classes: {svm_model.classes_}")

## 7. Model Evaluation and Threshold Tuning

In [ ]:
# Make predictions and get probabilities
y_pred_default = svm_model.predict(X_test_scaled)
y_proba = svm_model.predict_proba(X_test_scaled)[:, 1]

print(f"✓ Predictions generated")
print(f"  - Default threshold (0.5) predictions: {np.bincount(y_pred_default)}")
print(f"  - Probability range: [{y_proba.min():.4f}, {y_proba.max():.4f}]")

In [ ]:
# Evaluation with default threshold (0.5)
print("="*60)
print("SVM with DEFAULT THRESHOLD (0.5)")
print("="*60)
print(classification_report(y_test, y_pred_default, target_names=['No Churn', 'Churn']))

In [ ]:
# Apply custom threshold 0.65 (balanced precision-recall)
y_pred_065 = (y_proba >= 0.65).astype(int)

print("="*60)
print("SVM with CUSTOM THRESHOLD (0.65) - Balanced")
print("="*60)
print(classification_report(y_test, y_pred_065, target_names=['No Churn', 'Churn']))

In [ ]:
# Apply custom threshold 0.7 (higher precision, lower false positives)
y_pred_070 = (y_proba >= 0.7).astype(int)

print("="*60)
print("SVM with CUSTOM THRESHOLD (0.7) - Higher Precision")
print("="*60)
print(classification_report(y_test, y_pred_070, target_names=['No Churn', 'Churn']))

## 8. Confusion Matrix Visualization

In [ ]:
# Create images directory if it doesn't exist
os.makedirs("images", exist_ok=True)

# Generate confusion matrix using threshold 0.7
cm = confusion_matrix(y_test, y_pred_070)

# Create figure and plot
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=True,
    ax=ax,
    xticklabels=['No Churn', 'Churn'],
    yticklabels=['No Churn', 'Churn'],
    annot_kws={'size': 14, 'weight': 'bold'}
)
ax.set_xlabel("Predicted Label", fontsize=12, fontweight='bold')
ax.set_ylabel("True Label", fontsize=12, fontweight='bold')
ax.set_title("Confusion Matrix - SVM with Threshold 0.7", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()

# Save figure
output_path = "images/final_model.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"✓ Confusion matrix saved to: {output_path}")

# Display plot
plt.show()

# Print confusion matrix statistics
print(f"\n{'='*60}")
print("CONFUSION MATRIX INTERPRETATION")
print(f"{'='*60}")
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives (TN):  {tn}  - Correctly predicted No Churn")
print(f"False Positives (FP): {fp}  - Incorrectly predicted Churn")
print(f"False Negatives (FN): {fn}  - Incorrectly predicted No Churn")
print(f"True Positives (TP):  {tp}  - Correctly predicted Churn")
print(f"{'='*60}")

## 9. Summary and Key Metrics

In [ ]:
# Calculate key metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

metrics = {
    'Accuracy': accuracy_score(y_test, y_pred_070),
    'Precision': precision_score(y_test, y_pred_070),
    'Recall': recall_score(y_test, y_pred_070),
    'F1-Score': f1_score(y_test, y_pred_070)
}

print(f"\n{'='*60}")
print("FINAL MODEL PERFORMANCE (Threshold: 0.7)")
print(f"{'='*60}")
for metric, value in metrics.items():
    print(f"{metric:.<40} {value:.4f}")
print(f"{'='*60}")

print(f"\n✓ Model successfully trained and evaluated!")
print(f"✓ Results visualized and saved to images/final_model.png")

# Customer Churn Prediction
## Clean ML Pipeline with SVM Model

This notebook implements a complete machine learning pipeline to predict customer churn using Support Vector Machine (SVM) with SMOTE for handling class imbalance.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC

print("Libraries imported successfully")

## 2. Data Loading

In [ ]:
df = pd.read_csv("data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

## 3. Data Preprocessing

In [ ]:
# Remove customerID
df.drop("customerID", axis=1, inplace=True)

# Convert TotalCharges to numeric and handle missing values
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.dropna(inplace=True)

print(f"Dataset shape after cleaning: {df.shape}")
print(f"\nChurn distribution:")
print(df["Churn"].value_counts())

In [ ]:
# Encode categorical variables
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = le.fit_transform(df[col])

print("Encoding complete")
print(f"Dataset shape: {df.shape}")
df.head()

## 4. Train-Test Split

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

In [ ]:
# Scale features for SVM
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Features scaled successfully")

## 5. Data Balancing with SMOTE

In [ ]:
import subprocess
import sys

# Install imbalanced-learn
subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn", "-q"])

from imblearn.over_sampling import SMOTE

# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Original training set shape: {X_train.shape} with y shape {y_train.shape}")
print(f"Resampled training set shape: {X_train_sm.shape} with y shape {y_train_sm.shape}")

## 6. Model Training

In [ ]:
# Train SVM with class weighting for balanced predictions
svm_model = SVC(probability=True, class_weight='balanced', random_state=42)
svm_model.fit(X_train_sm, y_train_sm)

print("SVM model trained successfully")

## 7. Model Evaluation with Threshold Tuning

In [ ]:
# Default threshold (0.5)
svm_pred = svm_model.predict(X_test)

print("SVM with default threshold (0.5):\n")
print(classification_report(y_test, svm_pred))

In [ ]:
# Get probability predictions for threshold tuning
svm_proba = svm_model.predict_proba(X_test)[:, 1]

# Adjusted threshold 0.65 (balanced precision-recall)
threshold_65 = 0.65
svm_pred_65 = (svm_proba >= threshold_65).astype(int)

print("SVM with threshold 0.65 (Balanced):")
print(classification_report(y_test, svm_pred_65))

In [ ]:
# Adjusted threshold 0.7 (higher precision)
threshold_70 = 0.7
svm_pred_70 = (svm_proba >= threshold_70).astype(int)

print("SVM with threshold 0.7 (Higher Precision):")
print(classification_report(y_test, svm_pred_70))

In [ ]:
import os

# Create images directory if it doesn't exist
os.makedirs("images", exist_ok=True)

# Generate confusion matrix for final model (threshold 0.7)
cm = confusion_matrix(y_test, svm_pred_70)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True, ax=ax)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual", fontsize=12)
ax.set_title("Confusion Matrix - SVM with Threshold 0.7", fontsize=14, fontweight='bold')
plt.tight_layout()

# Save the figure as an image file
output_path = "images/final_model.png"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f"Confusion matrix saved to: {output_path}")

# Display the plot
plt.show()

# Print confusion matrix values
print(f"\nConfusion Matrix:\n{cm}")